In [142]:
import torch
from itertools import product

num_modes = 12
d_v = 64
x = torch.randn((d_v, 256, 256))

# Fourier transform
X = torch.fft.rfft2(x) # Outputfut of F(v_t)
X = X[:, :num_modes, :num_modes].permute(*torch.arange(x.ndim - 1, -1, -1))

R = torch.nn.Parameter(torch.randn(num_modes, num_modes, d_v, d_v)).to(torch.complex64)


In [70]:
R.shape

torch.Size([12, 12, 64, 64])

In [145]:
X.shape

torch.Size([12, 12, 64])

In [72]:
Y = torch.einsum('okij, okj -> oki', R, X)

In [95]:
k_max = 12
dv = 64
Fv = torch.randn((k_max, dv))
R = torch.randn((k_max, dv, dv))

In [ ]:
def apply_linear_simple(Fv, R):
    out = torch.empty_like(Fv)
    k = Fv.shape[0]
    l = Fv.shape[-1]

    for k_idx in range(k):
        for l_idx in range(l):
            out[k_idx, l_idx] = R[k_idx,l_idx, :] @ Fv[k_idx, :]

    return out
test = apply_linear_simple(Fv, R)
test_ein = torch.einsum('klj, kj -> kl', R, Fv)

print(Fv.shape, Fv.dtype)
print(R.shape, R.dtype)

a = apply_linear_simple(Fv, R)
b = torch.einsum('klm,km->kl', R, Fv)
print(a.shape, b.shape)
print((a - b).abs().max(), torch.allclose(a, b, atol=1e-5))

In [144]:
Fv = torch.randn((k_max, k_max, dv))
R = torch.randn((k_max, k_max, dv, dv))
def apply_linear_complicated(Fv, R):
    out = torch.empty_like(Fv)
    k = Fv.shape[0]
    l = R.shape[2]
    
    for k_tup in list(product(range(k), repeat=2)):
        k_idx1, k_idx2 = k_tup

        for l_idx in range(l):
            out[k_idx1, k_idx2, l_idx] = R[k_idx1, k_idx2, l_idx, :] @ Fv[k_idx1, k_idx2, :]

    return out

print(Fv.shape, Fv.dtype)
print(R.shape, R.dtype)

a = apply_linear_complicated(Fv, R)
b = torch.einsum('kxlj,kxj->kxl', R, Fv)
print(a.shape, b.shape)
print((a - b).abs().max(), torch.allclose(a, b, atol=1e-5))

torch.Size([12, 12, 64]) torch.float32
torch.Size([12, 12, 64, 64]) torch.float32
torch.Size([12, 12, 64]) torch.Size([12, 12, 64])
tensor(3.8147e-06) True


In [149]:
torch.fft.ifft2(b).shape

torch.Size([12, 12, 64])